In [2]:
from pathlib import Path
import os

from datasets import load_dataset
from huggingface_hub import login, snapshot_download

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)

repo_id = "villacu/cammt"
local_dir = Path("./cammt_data")

snapshot_download(repo_id=repo_id, repo_type="dataset", local_dir=str(local_dir))

# Clean up interrupted downloads, then re-download to get valid parquet files.
fileparts = list(local_dir.rglob("*.parquet.filepart"))
if fileparts:
    for fp in fileparts:
        fp.unlink()
    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=str(local_dir),
        force_download=True,
    )

parquet_files = sorted(str(p) for p in local_dir.rglob("*.parquet"))
if not parquet_files:
    raise RuntimeError(
        "No .parquet files found in ./cammt_data after download. "
        "Check network/auth and try again."
    )

ds = load_dataset("parquet", data_files={"train": parquet_files})
print(ds)

Fetching 25 files: 100%|██████████| 25/25 [00:07<00:00,  3.21it/s]
Generating train split: 5817 examples [00:08, 670.33 examples/s]


DatasetDict({
    train: Dataset({
        features: ['ID', 'regional', 'English', 'Conserved_translation', 'Substituted_translation', 'Category', 'Preferred_translation', 'image'],
        num_rows: 5817
    })
})


## US Prediction Analysis

Find all (split, image_id) triplets across `image_only` / `image_english` / `image_regional` where **at least one** experiment predicted a US variant ("United States", "United States of America", "USA"). Save those images to `us_prediction_images/`.

In [1]:
import io
import json
import math
import re
from collections import defaultdict, Counter
from pathlib import Path

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

EXPERIMENTS = ["image_only", "image_english", "image_regional"]
RESULTS_DIR = Path("results")
DATA_DIR = Path("cammt_data/data")
SAVE_DIR = Path("us_prediction_images")

US_VARIANTS = re.compile(r"^(United States(?: of America)?|USA)$", re.IGNORECASE)

def is_us(pred: str) -> bool:
    return bool(US_VARIANTS.match(pred.strip())) if pred else False

def load_all_results() -> dict:
    """Returns {split: {id: {experiment: record}}}"""
    data = defaultdict(lambda: defaultdict(dict))
    for exp in EXPERIMENTS:
        exp_dir = RESULTS_DIR / exp
        if not exp_dir.exists():
            print(f"Warning: {exp_dir} not found, skipping")
            continue
        for jsonl_path in sorted(exp_dir.glob("*.jsonl")):
            with open(jsonl_path) as f:
                for line in f:
                    r = json.loads(line)
                    data[r["split"]][r["id"]][exp] = r
    return data

def find_us_triplets(data: dict) -> list:
    """Return list of (split, id, exps_dict) where ≥1 experiment predicted a US variant.
    Only includes IDs present in all 3 experiments."""
    triplets = []
    for split, ids in data.items():
        for rid, exps in ids.items():
            if len(exps) < 3:
                continue
            preds = {e: exps[e]["predicted"] for e in EXPERIMENTS}
            if any(is_us(p) for p in preds.values()):
                triplets.append((split, rid, exps))
    return triplets

print("Loading results...")
data = load_all_results()

print("Finding US triplets...")
triplets = find_us_triplets(data)
print(f"Total triplets with ≥1 US prediction: {len(triplets)}")

# Breakdown by split
split_counts = Counter(t[0] for t in triplets)
print("\nBreakdown by split:")
for split, count in sorted(split_counts.items(), key=lambda x: -x[1]):
    print(f"  {split:20s}: {count}")

# Breakdown by experiment
exp_counts = Counter()
for _, _, exps in triplets:
    for exp in EXPERIMENTS:
        rec = exps.get(exp)
        if rec and is_us(rec["predicted"]):
            exp_counts[exp] += 1
print("\nUS predictions per experiment:")
for exp in EXPERIMENTS:
    print(f"  {exp:20s}: {exp_counts.get(exp, 0)}")

Loading results...
Finding US triplets...
Total triplets with ≥1 US prediction: 494

Breakdown by split:
  bg_bg               : 60
  es_chl              : 55
  es_arg              : 49
  pt_brz              : 41
  ru_rus              : 34
  jp_jap              : 26
  ar_egy              : 23
  es_mex              : 23
  ms_mys              : 23
  om_eth              : 23
  es_ecu              : 19
  fil_phl             : 19
  kor_sk              : 19
  ig_nga              : 16
  amh_eth             : 14
  sw_ken              : 10
  ur_pak              : 9
  bn_india            : 8
  ind_ind             : 8
  mr_india            : 4
  ta_india            : 4
  ur_india            : 4
  zh_ch               : 3

US predictions per experiment:
  image_only          : 433
  image_english       : 172
  image_regional      : 27


In [2]:
def load_parquet_index(split: str) -> dict:
    parquet_path = DATA_DIR / f"{split}-00000-of-00001.parquet"
    df = pd.read_parquet(parquet_path, columns=["ID", "image", "English", "Category"])
    return df.set_index("ID").to_dict("index")

def get_pil_image(row_data: dict) -> Image.Image:
    img_field = row_data["image"]
    if isinstance(img_field, dict):
        raw = img_field.get("bytes") or img_field.get("path")
        if isinstance(raw, bytes):
            return Image.open(io.BytesIO(raw)).convert("RGB")
    if isinstance(img_field, bytes):
        return Image.open(io.BytesIO(img_field)).convert("RGB")
    raise ValueError(f"Unexpected image format: {type(img_field)}")

def pred_color(pred: str, gt: str) -> str:
    if is_us(pred):
        return "red"
    if pred.strip().lower() == gt.strip().lower():
        return "green"
    return "orange"

SAVE_DIR.mkdir(parents=True, exist_ok=True)
parquet_cache = {}

print(f"Saving {len(triplets)} images to {SAVE_DIR}/...")
for split, rid, exps in triplets:
    if split not in parquet_cache:
        parquet_cache[split] = load_parquet_index(split)
    row_data = parquet_cache[split].get(rid)
    if row_data is None:
        print(f"  Warning: {rid} not found in {split} parquet")
        continue
    img = get_pil_image(row_data)
    img.save(SAVE_DIR / f"{split}__{rid}.jpg")

print(f"Done. {len(triplets)} images saved.")
saved = list(SAVE_DIR.glob("*.jpg"))
print(f"Files in {SAVE_DIR}: {len(saved)}")

Saving 494 images to us_prediction_images/...
Done. 494 images saved.
Files in us_prediction_images: 494


In [ ]:
from collections import defaultdict

# Group triplets by split
by_split = defaultdict(list)
for split, rid, exps in triplets:
    by_split[split].append((rid, exps))

# Load regional text too — need all columns this time
def load_parquet_full(split: str) -> dict:
    parquet_path = DATA_DIR / f"{split}-00000-of-00001.parquet"
    df = pd.read_parquet(parquet_path, columns=["ID", "English", "regional", "Category"])
    return df.set_index("ID").to_dict("index")

parquet_full_cache = {}

for split, items in sorted(by_split.items()):
    if split not in parquet_full_cache:
        parquet_full_cache[split] = load_parquet_full(split)
    index = parquet_full_cache[split]

    out_path = SAVE_DIR / f"{split}.jsonl"
    with open(out_path, "w") as f:
        for rid, exps in items:
            row = index.get(rid, {})
            record = {
                "id": rid,
                "split": split,
                "ground_truth": exps[EXPERIMENTS[0]]["ground_truth"],
                "category": row.get("Category", ""),
                "english": row.get("English", ""),
                "regional": row.get("regional", ""),
                "predictions": {
                    exp: {
                        "predicted": exps[exp]["predicted"],
                        "correct": exps[exp]["correct"],
                        "is_us": is_us(exps[exp]["predicted"]),
                        "raw_output": exps[exp]["raw_output"],
                    }
                    for exp in EXPERIMENTS
                },
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"  {out_path}  ({len(items)} records)")

print(f"\nJSONL files written to {SAVE_DIR}/")

In [ ]:
COLS = 4
SHORT = {"image_only": "img_only", "image_english": "img_en", "image_regional": "img_reg"}

n = len(triplets)
rows = math.ceil(n / COLS)
fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 4.5, rows * 5.5))
axes = axes.flatten() if n > 1 else [axes]

for idx, (split, rid, exps) in enumerate(triplets):
    ax = axes[idx]
    row_data = parquet_cache.get(split, {}).get(rid)
    if row_data is None:
        ax.set_visible(False)
        continue

    img = get_pil_image(row_data)
    ax.imshow(img)
    ax.axis("off")

    gt = exps[EXPERIMENTS[0]]["ground_truth"]
    category = row_data.get("Category", "")
    english = row_data.get("English", "") or ""
    if len(english) > 70:
        english = english[:67] + "..."

    ax.set_title(f"GT: {gt} | {category}\n{english}", fontsize=6.5, pad=3)

    y = -0.03
    for exp in EXPERIMENTS:
        rec = exps.get(exp)
        pred = rec["predicted"] if rec else "N/A"
        marker = " ★" if is_us(pred) else ""
        label = f"[{SHORT[exp]}] {pred}{marker}"
        ax.text(0.5, y, label, transform=ax.transAxes,
                ha="center", va="top", fontsize=6,
                color=pred_color(pred, gt), clip_on=False)
        y -= 0.07

for ax in axes[n:]:
    ax.set_visible(False)

legend_elements = [
    mpatches.Patch(color="red",    label="US prediction (★)"),
    mpatches.Patch(color="green",  label="Correct prediction"),
    mpatches.Patch(color="orange", label="Wrong (non-US) prediction"),
]
fig.legend(handles=legend_elements, loc="lower right", fontsize=8)
fig.suptitle(f"Triplets with ≥1 US prediction  (n={n})", fontsize=12, fontweight="bold")
plt.tight_layout()

grid_path = SAVE_DIR / "grid_overview.png"
plt.savefig(grid_path, bbox_inches="tight", dpi=150)
print(f"Grid saved to {grid_path}")
plt.show()